# Churn Prediction — Modular ML Pipeline
This notebook trains and evaluates Logistic Regression, XGBoost, and LightGBM classifiers on a customer churn dataset using Optuna for hyperparameter tuning.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.metrics import (
    classification_report, roc_auc_score, auc, roc_curve
)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.base import clone

from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

## 2. Configuration

In [ ]:
# ── Global settings ───────────────────────────────────────────────────────────
RANDOM_STATE = 30
N_SPLITS     = 5
N_TRIALS     = 50
SCORING      = 'recall'
TEST_SIZE    = 0.2

TRAIN_PATH   = '../data/train.csv'
TEST_PATH    = '../data/test.csv'

OHE_COLS     = ['Subscription Type', 'Contract Length', 'Gender']
INT_COLS     = ['Age', 'Usage Frequency', 'Support Calls',
                'Payment Delay', 'Last Interaction', 'Churn']
FLOAT_COLS   = ['Tenure', 'Total Spend']

COLORS = {
    'Logistic Regression': '#4A90D9',
    'XGBoost'            : '#E74C3C',
    'LightGBM'           : '#F39C12',
}

## 3. Data Loading & Cleaning

In [ ]:
def load_data(train_path: str, test_path: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Load and perform basic cleaning on train/test CSVs."""
    train = pd.read_csv(train_path).drop(columns='CustomerID').dropna()
    test  = pd.read_csv(test_path).drop(columns='CustomerID')
    return train, test


def cast_dtypes(train: pd.DataFrame, test: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Enforce correct dtypes for known columns."""
    train[INT_COLS]   = train[INT_COLS].astype(int)
    test[FLOAT_COLS]  = test[FLOAT_COLS].astype(float)
    return train, test


def data_quality_report(df: pd.DataFrame, label: str = '') -> None:
    """Print missing values and duplicate count."""
    print(f'── {label} ──')
    print('Missing values:\n', df.isna().sum())
    print(f'Duplicates: {df.duplicated().sum()}\n')


# ── Run ───────────────────────────────────────────────────────────────────────
train_df, test_df = load_data(TRAIN_PATH, TEST_PATH)
data_quality_report(train_df, 'Train')
data_quality_report(test_df,  'Test')

train_df, test_df = cast_dtypes(train_df, test_df)
train_df.info()

## 4. Exploratory Data Analysis

In [ ]:
def plot_distributions(df: pd.DataFrame) -> None:
    """Plot a histogram for every column in df."""
    for col in df.columns:
        fig, ax = plt.subplots(figsize=(10, 5))
        sns.histplot(df[col], color='grey', edgecolor='black', bins=20, ax=ax)
        ax.set_title(f'{col} Distribution')
        plt.tight_layout()
        plt.show()


combined_df = pd.concat([train_df, test_df], axis=0)
plot_distributions(combined_df)

## 5. Preprocessing & Feature Engineering

In [ ]:
def build_preprocessor(df: pd.DataFrame) -> ColumnTransformer:
    """Build a ColumnTransformer that OHE-encodes categoricals and scales numerics."""
    num_cols = df.select_dtypes('number').drop(columns='Churn', errors='ignore').columns.tolist()
    return ColumnTransformer(transformers=[
        ('ohe',    OneHotEncoder(drop='first', sparse_output=False), OHE_COLS),
        ('scaler', MinMaxScaler(),                                   num_cols),
    ], remainder='passthrough')


def compute_vif(X: pd.DataFrame) -> pd.DataFrame:
    """Return VIF scores for all numeric features (excluding the constant)."""
    X_num      = X.select_dtypes(include=[np.number])
    X_const    = add_constant(X_num)
    vif        = pd.DataFrame()
    vif['Feature'] = X_const.columns
    vif['VIF']     = [variance_inflation_factor(X_const.values, i)
                      for i in range(X_const.shape[1])]
    return vif[vif['Feature'] != 'const'].sort_values('VIF', ascending=False)


# ── Run ───────────────────────────────────────────────────────────────────────
X = combined_df.drop(columns='Churn')
y = combined_df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)
print(f'X_train: {X_train.shape} | X_test: {X_test.shape}')

preprocessor = build_preprocessor(combined_df)
skf          = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

print('\nVIF Scores:')
print(compute_vif(X_train))

## 6. Model Training with Optuna

In [ ]:
def make_pipeline(preprocessor: ColumnTransformer, model) -> Pipeline:
    """Wrap a preprocessor + classifier in a sklearn Pipeline."""
    return Pipeline([
        ('scaler/encoder', clone(preprocessor)),
        ('classifier',     model),
    ])


def cv_score(pipeline: Pipeline, X, y, skf, scoring: str) -> float:
    """Return mean CV score penalised by half a standard deviation."""
    scores = cross_val_score(pipeline, X, y, cv=skf, scoring=scoring, n_jobs=-1)
    return scores.mean() - (scores.std() * 0.5)


def fit_and_evaluate(pipeline: Pipeline, X_train, y_train, X_test, y_test) -> tuple:
    """Fit a pipeline and return (predictions, probabilities)."""
    pipeline.fit(X_train, y_train)
    pred  = pipeline.predict(X_test)
    proba = pipeline.predict_proba(X_test)[:, 1]
    print(classification_report(y_test, pred, digits=2))
    return pred, proba


test_proba: dict = {}

### 6a. Logistic Regression

In [ ]:
def objective_lr(trial) -> float:
    params = {
        'C'           : trial.suggest_float('C', 0.001, 10.0, log=True),
        'max_iter'    : trial.suggest_int('max_iter', 200, 1000),
        'class_weight': trial.suggest_categorical('class_weight', [None, 'balanced']),
    }
    pipeline = make_pipeline(preprocessor, LogisticRegression(**params, random_state=RANDOM_STATE))
    return cv_score(pipeline, X_train, y_train, skf, SCORING)


study_lr = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
)
study_lr.optimize(objective_lr, n_trials=N_TRIALS, show_progress_bar=True)
print(f'Best CV {SCORING}: {study_lr.best_value:.3f} | Params: {study_lr.best_params}')

lr_best = make_pipeline(
    preprocessor,
    LogisticRegression(**study_lr.best_params, random_state=RANDOM_STATE)
)
_, test_proba['Logistic Regression'] = fit_and_evaluate(lr_best, X_train, y_train, X_test, y_test)

### 6b. XGBoost

In [ ]:
def objective_xgb(trial) -> float:
    params = {
        'n_estimators'    : trial.suggest_int('n_estimators', 200, 800),
        'max_depth'       : trial.suggest_int('max_depth', 3, 7),
        'learning_rate'   : trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample'       : trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha'       : trial.suggest_float('reg_alpha', 0.0001, 10.0, log=True),
        'reg_lambda'      : trial.suggest_float('reg_lambda', 0.0001, 10.0, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 10.0),
    }
    pipeline = make_pipeline(
        preprocessor,
        XGBClassifier(**params, random_state=RANDOM_STATE, eval_metric='logloss', verbosity=0, n_jobs=-1)
    )
    return cv_score(pipeline, X_train, y_train, skf, SCORING)


study_xgb = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
)
study_xgb.optimize(objective_xgb, n_trials=N_TRIALS, show_progress_bar=True)
print(f'Best CV {SCORING}: {study_xgb.best_value:.3f} | Params: {study_xgb.best_params}')

xgb_best = make_pipeline(
    preprocessor,
    XGBClassifier(**study_xgb.best_params, random_state=RANDOM_STATE,
                  eval_metric='logloss', verbosity=0, n_jobs=-1)
)
_, test_proba['XGBoost'] = fit_and_evaluate(xgb_best, X_train, y_train, X_test, y_test)

### 6c. LightGBM

In [ ]:
def objective_lgbm(trial) -> float:
    params = {
        'n_estimators'    : trial.suggest_int('n_estimators', 100, 500),
        'max_depth'       : trial.suggest_int('max_depth', 3, 12),
        'learning_rate'   : trial.suggest_float('learning_rate', 0.001, 0.3, log=True),
        'num_leaves'      : trial.suggest_int('num_leaves', 20, 150),
        'subsample'       : trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha'       : trial.suggest_float('reg_alpha', 0.0001, 10.0, log=True),
        'reg_lambda'      : trial.suggest_float('reg_lambda', 0.0001, 10.0, log=True),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 10.0),
    }
    pipeline = make_pipeline(
        preprocessor,
        LGBMClassifier(**params, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)
    )
    return cv_score(pipeline, X_train, y_train, skf, SCORING)


study_lgbm = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
)
study_lgbm.optimize(objective_lgbm, n_trials=N_TRIALS, show_progress_bar=True)
print(f'Best CV {SCORING}: {study_lgbm.best_value:.3f} | Params: {study_lgbm.best_params}')

lgbm_best = make_pipeline(
    preprocessor,
    LGBMClassifier(**study_lgbm.best_params, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)
)
_, test_proba['LightGBM'] = fit_and_evaluate(lgbm_best, X_train, y_train, X_test, y_test)

## 7. Evaluation & Visualisation

In [ ]:
def plot_roc_comparison(test_proba: dict, y_test, colors: dict) -> None:
    """Side-by-side ROC curves and AUC leaderboard bar chart."""
    fig, (ax_roc, ax_bar) = plt.subplots(1, 2, figsize=(16, 7))
    fig.suptitle('ROC-AUC Comparison: Tuned Models', fontsize=16, fontweight='bold', y=1.02)

    ax_roc.plot([0, 1], [0, 1], 'k--', lw=1.2, label='Random classifier', alpha=0.5)

    rows = []
    for name, proba in test_proba.items():
        fpr, tpr, _ = roc_curve(y_test, proba)
        score = auc(fpr, tpr)
        rows.append({'Model': name, 'AUC': score})
        sns.lineplot(x=fpr, y=tpr, ax=ax_roc, lw=2.5,
                     color=colors.get(name, '#333333'),
                     label=f'{name} (AUC = {score:.4f})')

    ax_roc.set(title='ROC Curves', xlabel='False Positive Rate', ylabel='True Positive Rate')
    ax_roc.grid(alpha=0.3)
    ax_roc.legend(loc='lower right')

    df_aucs = pd.DataFrame(rows).sort_values('AUC', ascending=False)
    sns.barplot(data=df_aucs, x='AUC', y='Model', palette=colors,
                ax=ax_bar, hue='Model', legend=False, alpha=0.9, edgecolor='white')

    for i, val in enumerate(df_aucs['AUC']):
        ax_bar.text(val - 0.005, i, f'{val:.4f}', va='center', ha='right',
                    color='white', fontweight='bold', fontsize=11)

    ax_bar.set_xlim([max(0, df_aucs['AUC'].min() - 0.05), 1.0])
    ax_bar.set(title='AUC Score Comparison (Leaderboard)', xlabel='ROC-AUC Score', ylabel='')

    plt.tight_layout()
    plt.show()

    # ── Leaderboard printout ──────────────────────────────────────────────────
    print('\n' + '═' * 45)
    print(f"{'RANK':<5} {'MODEL':<25} {'TEST AUC':<10}")
    print('─' * 45)
    for rank, row in enumerate(df_aucs.itertuples(), 1):
        star = '⭐' if rank == 1 else '  '
        print(f'{rank:<5} {row.Model:<25} {row.AUC:<10.4f} {star}')
    print('═' * 45)


plot_roc_comparison(test_proba, y_test, COLORS)

## 8. Feature Importance

In [ ]:
def plot_feature_importance(
    importances: np.ndarray,
    feature_names: list[str],
    title: str,
    color: str,
    top_n: int = 20,
) -> None:
    """Horizontal bar chart of the top-N most important features."""
    df_imp = (
        pd.DataFrame({'feature': feature_names, 'importance': importances})
        .reindex(pd.Series(importances).abs().sort_values(ascending=False).index)
        .head(top_n)
    )

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(df_imp['feature'][::-1], df_imp['importance'][::-1], color=color)
    ax.set(title=title, xlabel='Importance / Coefficient Value', ylabel='Feature')
    plt.tight_layout()
    plt.show()


feature_names = preprocessor.get_feature_names_out().tolist()

importance_configs = [
    (lr_best,   lr_best.named_steps['classifier'].coef_[0],             'Logistic Regression — Feature Coefficients', '#4A90D9'),
    (xgb_best,  xgb_best.named_steps['classifier'].feature_importances_, 'XGBoost — Feature Importance',               '#E74C3C'),
    (lgbm_best, lgbm_best.named_steps['classifier'].feature_importances_, 'LightGBM — Feature Importance',              '#F39C12'),
]

for _, importances, title, color in importance_configs:
    plot_feature_importance(importances, feature_names, title, color)